# Lesson 1 · Simple harmonic motion

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/StativeSh/quantum-playground/blob/main/lessons/01_simple_harmonic_motion.ipynb)

**For:** A-level Physics (for example AQA 3.6.1.2 to 3.6.1.4, OCR A 5.3, Edexcel Topic 13) and IB Physics (C.1, with resonance from C.4). Check your own specification for the exact wording.

**You will be able to:**

1. state the defining equation of SHM and use $x = A\cos\omega t$, $v = \pm\omega\sqrt{A^2 - x^2}$ and $a = -\omega^2 x$
2. explain the phase differences between displacement, velocity and acceleration
3. use $T = 2\pi\sqrt{m/k}$ and $T = 2\pi\sqrt{l/g}$, and say when the pendulum formula stops working
4. describe the energy changes in SHM
5. distinguish light, critical and heavy damping, and explain resonance

**How to use this lesson:** run each cell with Shift+Enter. Where you see *Try it*, change a number and run the cell again. Exercises are at the end, with worked solutions below them.

In [ ]:
# Setup: installs the qplay package if it is missing (for example on Google Colab, about 20 s).
import importlib.util, subprocess, sys
if importlib.util.find_spec("qplay") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/StativeSh/quantum-playground"], check=True)

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from qplay import mechanics, numerics

## 1. What makes motion "simple harmonic"?

An object moves with SHM when its acceleration is **proportional to its displacement** and **directed towards equilibrium**:

$$a = -\omega^2 x$$

The minus sign is the whole idea: the further you pull it, the harder it is pulled back. Solving this gives, for release from rest at amplitude $A$,

$$x = A\cos\omega t,\qquad v = -\omega A\sin\omega t,\qquad a = -\omega^2 A\cos\omega t,\qquad T = \frac{2\pi}{\omega}.$$

Below, the computer solves $a = -\omega^2x$ step by step, with no formula for $x(t)$ built in, and we compare it with $A\cos\omega t$.

In [ ]:
A, omega = 0.05, 4.0          # amplitude in m, angular frequency in rad/s   <- Try it: change these
t, x, v = mechanics.harmonic_oscillator(x0=A, v0=0, omega=omega, t_max=2 * 2 * np.pi / omega, n=600)
a = -omega**2 * x

fig, axes = plt.subplots(3, 1, figsize=(7, 6), sharex=True)
axes[0].plot(t, x); axes[0].plot(t, A * np.cos(omega * t), "k:", label=r"$A\cos\omega t$"); axes[0].set_ylabel("x / m"); axes[0].legend(loc="upper right")
axes[1].plot(t, v, color="C1"); axes[1].set_ylabel("v / m s$^{-1}$")
axes[2].plot(t, a, color="C2"); axes[2].set_ylabel("a / m s$^{-2}$"); axes[2].set_xlabel("t / s")
for ax in axes: ax.axhline(0, color="0.7", lw=0.6)
print(f"T = {2 * np.pi / omega:.3f} s,  v_max = omega*A = {omega * A:.3f} m/s,  a_max = omega^2*A = {omega**2 * A:.3f} m/s^2")

**Read the graphs.**

- When $x$ is at a maximum, $v$ is zero and $a$ is at its most negative.
- Velocity is a quarter of a cycle ($\pi/2$) ahead of displacement.
- Acceleration is exactly out of phase ($\pi$) with displacement: it is the displacement graph flipped and scaled by $\omega^2$.

Exam tip: the velocity at any displacement, without needing the time, is $v = \pm\omega\sqrt{A^2 - x^2}$. The next cell checks that against the simulation.

In [ ]:
plt.figure(figsize=(5, 4))
plt.plot(x, v, label="simulation")
xs = np.linspace(-A, A, 200)
plt.plot(xs, omega * np.sqrt(A**2 - xs**2), "k:", label=r"$\pm\omega\sqrt{A^2-x^2}$"); plt.plot(xs, -omega * np.sqrt(A**2 - xs**2), "k:")
plt.xlabel("x / m"); plt.ylabel("v / m s$^{-1}$"); plt.legend(); plt.title("Velocity against displacement is an ellipse");

## 2. Two systems you must know

**Mass on a spring:** $T = 2\pi\sqrt{\dfrac{m}{k}}$ &nbsp;&nbsp; **Simple pendulum:** $T = 2\pi\sqrt{\dfrac{l}{g}}$

Neither period depends on amplitude. For the spring that is exactly true. For the pendulum it is only true for **small angles**, because the derivation uses $\sin\theta\approx\theta$. How small is small? The package can compute the exact period at any amplitude.

In [ ]:
l, g = 1.0, 9.81
T0 = 2 * np.pi * np.sqrt(l / g)
degrees = np.array([5, 10, 20, 30, 45, 60, 90])
exact = mechanics.pendulum_period(np.radians(degrees), g, l)
print(f"Formula: T = {T0:.4f} s\n")
print("amplitude   exact T    error in formula")
for d, T in zip(degrees, exact):
    print(f"{d:6d} deg   {T:.4f} s   {100 * (T - T0) / T:5.2f} %")

At $10^\circ$ the formula is off by about 0.2%, which is why "small angle" in practical work usually means under about $10^\circ$. By $45^\circ$ it is 4% out, easily enough to spoil a measurement of $g$.

## 3. Energy in SHM

With no damping, total energy is constant and swaps between kinetic and potential:

$$E_k = \tfrac12 m v^2,\qquad E_p = \tfrac12 k x^2,\qquad E_{total} = \tfrac12 k A^2 = \tfrac12 m\omega^2A^2 .$$

In [ ]:
m, k = 0.25, 40.0             # kg, N/m
w = np.sqrt(k / m)
t, x, v = mechanics.harmonic_oscillator(x0=0.05, v0=0, omega=w, t_max=2 * np.pi / w, n=400)
Ek, Ep = 0.5 * m * v**2, 0.5 * k * x**2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.6))
ax1.plot(t, Ek, label="kinetic"); ax1.plot(t, Ep, label="potential"); ax1.plot(t, Ek + Ep, "k--", label="total")
ax1.set_xlabel("t / s"); ax1.set_ylabel("energy / J"); ax1.legend(); ax1.set_title("Against time")
ax2.plot(x, Ek, label="kinetic"); ax2.plot(x, Ep, label="potential"); ax2.plot(x, Ek + Ep, "k--")
ax2.set_xlabel("x / m"); ax2.set_title("Against displacement"); ax2.legend();

Notice that the energies oscillate at **twice** the frequency of the motion: kinetic energy peaks every time the mass passes through equilibrium, in either direction. Against displacement, $E_p$ is a parabola and $E_k$ is the same parabola upside down.

## 4. Damping

Real oscillators lose energy. With a resistive force proportional to velocity:

- **Light damping:** it still oscillates, with amplitude decaying exponentially and the period almost unchanged.
- **Critical damping:** it returns to equilibrium in the shortest time without oscillating (car suspension, door closers).
- **Heavy damping:** it creeps back slowly without oscillating.

In [ ]:
w0 = 2 * np.pi                # natural angular frequency: period 1 s
plt.figure(figsize=(7, 4))
for gamma, label in [(0.3, "light"), (w0, "critical"), (4 * w0, "heavy")]:
    t, x, _ = mechanics.harmonic_oscillator(x0=1, v0=0, omega=w0, gamma=gamma, t_max=4, n=800)
    plt.plot(t, x, label=label)
plt.plot(t, np.exp(-0.3 * t), "k:", lw=0.8, label="exponential envelope"); plt.plot(t, -np.exp(-0.3 * t), "k:", lw=0.8)
plt.axhline(0, color="0.7", lw=0.6); plt.xlabel("t / s"); plt.ylabel("displacement"); plt.legend();

## 5. Forced oscillations and resonance

Drive an oscillator with a periodic force and it eventually oscillates at the **driving** frequency. Its amplitude depends on how close that is to the **natural** frequency $f_0$: when they match, energy is transferred most efficiently and the amplitude peaks. That is **resonance**.

Each dot below is a separate simulation, run long enough to settle. The curves are the theoretical steady-state amplitude.

In [ ]:
w0, F = 1.0, 1.0
drive = np.linspace(0.2, 2.0, 300)
plt.figure(figsize=(7, 4.2))
for i, gamma in enumerate([0.1, 0.25, 0.6]):
    plt.plot(drive, F / np.sqrt((w0**2 - drive**2) ** 2 + (2 * gamma * drive) ** 2), color=f"C{i}", label=f"damping {gamma}")
    for wd in [0.5, 0.8, 1.0, 1.2, 1.6]:
        t, x, _ = mechanics.harmonic_oscillator(x0=0, v0=0, omega=w0, gamma=gamma, drive=F, drive_omega=wd, t_max=120, n=3000)
        plt.plot(wd, np.abs(x[t > 90]).max(), "o", color=f"C{i}", ms=5)
plt.axvline(w0, color="0.6", lw=0.6, ls="--"); plt.xlabel("driving frequency / natural frequency"); plt.ylabel("steady-state amplitude"); plt.legend();

**What to say in the exam.** As damping increases:

- the resonance peak is **lower**
- the peak is **broader**
- the peak shifts to a frequency **slightly below** the natural frequency

Lightly damped systems have sharp, tall resonances (a wine glass, a tuning fork); heavily damped ones barely resonate (a car with good shock absorbers).

---

## Exercises

Work on paper first, then put your answer in the cell and run it. Take $g = 9.81\ \mathrm{m\,s^{-2}}$. Give answers to 3 significant figures.

**Q1.** A 0.25 kg mass hangs on a spring of stiffness 40 N m$^{-1}$ and oscillates with amplitude 0.050 m. Find (a) the period, (b) the maximum speed, (c) the maximum acceleration.

**Q2.** For the oscillator in Q1, find the speed when the displacement is 0.030 m.

**Q3.** A pendulum has a period of 2.00 s for small oscillations. Find its length.

**Q4.** At what displacement, as a fraction of the amplitude, are the kinetic and potential energies equal?

**Q5 (simulation).** Use `mechanics.pendulum_period` to find the amplitude, in degrees, at which $T = 2\pi\sqrt{l/g}$ underestimates the true period by 1.0%.

In [ ]:
def check(name, yours, correct, tol=0.01):
    if yours is None:
        print(f"{name}: not answered yet")
    elif abs(yours - correct) <= tol * abs(correct):
        print(f"{name}: correct")
    else:
        print(f"{name}: not yet, check your working (you gave {yours})")

# The correct values are computed here from the physics, not typed in.
_m, _k, _A = 0.25, 40.0, 0.050
_w = np.sqrt(_k / _m)
_answers = {
    "Q1a": 2 * np.pi / _w, "Q1b": _w * _A, "Q1c": _w**2 * _A,
    "Q2": _w * np.sqrt(_A**2 - 0.030**2),
    "Q3": 9.81 * 2.00**2 / (4 * np.pi**2),
    "Q4": 1 / np.sqrt(2),
    "Q5": np.degrees(numerics.bisect(lambda th: (mechanics.pendulum_period(th) - 2 * np.pi * np.sqrt(1 / 9.81)) / mechanics.pendulum_period(th) - 0.010, 0.01, 1.5)),
}

# Replace None with your answers, in SI units (Q4 as a decimal fraction, Q5 in degrees).
mine = {"Q1a": None, "Q1b": None, "Q1c": None, "Q2": None, "Q3": None, "Q4": None, "Q5": None}
for q in mine:
    check(q, mine[q], _answers[q])

---

## Worked solutions

Try the questions before reading on.

**Q1.** $\omega = \sqrt{k/m} = \sqrt{40/0.25} = 12.6\ \mathrm{rad\,s^{-1}}$.
(a) $T = 2\pi/\omega = 0.497\ \mathrm{s}$ &nbsp; (b) $v_{max} = \omega A = 0.632\ \mathrm{m\,s^{-1}}$ &nbsp; (c) $a_{max} = \omega^2 A = 160 \times 0.050 = 8.00\ \mathrm{m\,s^{-2}}$

**Q2.** $v = \omega\sqrt{A^2 - x^2} = 12.65\sqrt{0.050^2 - 0.030^2} = 12.65 \times 0.040 = 0.506\ \mathrm{m\,s^{-1}}$

**Q3.** $T = 2\pi\sqrt{l/g} \Rightarrow l = \dfrac{gT^2}{4\pi^2} = \dfrac{9.81 \times 4.00}{39.48} = 0.994\ \mathrm{m}$

**Q4.** $\tfrac12 kx^2 = \tfrac12 \times \tfrac12 kA^2 \Rightarrow x = A/\sqrt2 = 0.707A$. Not halfway: the potential energy grows with $x^2$.

**Q5.** The cell below finds it numerically.

In [ ]:
for q, value in _answers.items():
    print(f"{q}: {value:.3g}")

The answer to Q5 is about $23^\circ$. A handy approximation is $T \approx T_0\,(1 + \theta_0^2/16)$ with $\theta_0$ in radians, which gives 1% at $\theta_0 = 0.4\ \mathrm{rad} \approx 23^\circ$.

**Next:** try changing the damping in section 5 to 0.02 and see how sharp the resonance becomes.